# Step 10: Time & Velocity Analysis

This notebook performs recruitment cycle-time and hiring velocity analytics to:
1. Calculate average, median, standard deviation, and IQR of days spent at each recruitment stage.
2. Identify process bottlenecks and stages with extreme latency.
3. Compare overall Time-to-Hire (joined candidates) against Time-to-Drop (dropped candidates).
4. Drill down into department-level and role-level hiring velocity.
5. Highlight delayed drop-offs (candidates who stay long in the pipeline before dropping off).

In [ ]:
import os
import pandas as pd
import numpy as np

# Path setup
FEATURES_PATH = os.path.join("..", "data", "processed", "candidate_features.csv")
STAGES_PATH = os.path.join("..", "data", "processed", "recruitment_stages.csv")
print(f"Checking features file at: {FEATURES_PATH}")

## Load Data

We load the engineered candidate features dataset.

In [ ]:
df = pd.read_csv(FEATURES_PATH)
df.head()

## Part 1: Stage-wise Duration Distribution

We compute the mean, median, min, max, std dev, and 75th percentile of duration for candidates who passed through each stage.

In [ ]:
STAGE_ORDER = ["Application", "Screening", "Interview", "Offer", "Joined"]

stage_metrics = []
for idx, stage in enumerate(STAGE_ORDER):
    dur_col = f"duration_{stage.lower()}_days"
    active_candidates = df[df['furthest_stage_index'] >= idx]
    count = len(active_candidates)
    
    if count > 0 and dur_col in df.columns:
        durs = active_candidates[dur_col].dropna()
        avg_d = round(float(durs.mean()), 2)
        med_d = round(float(durs.median()), 2)
        min_d = int(durs.min()) if not durs.empty else 0
        max_d = int(durs.max()) if not durs.empty else 0
        std_d = round(float(durs.std()), 2) if len(durs) > 1 else 0.0
        p75_d = round(float(durs.quantile(0.75)), 2) if not durs.empty else 0.0
    else:
        avg_d = med_d = std_d = p75_d = 0.0
        min_d = max_d = 0
        
    stage_metrics.append({
        'stage': stage,
        'candidates_count': count,
        'avg_days': avg_d,
        'median_days': med_d,
        'min_days': min_d,
        'max_days': max_d,
        'std_days': std_d,
        'p75_days': p75_d
    })

df_stages = pd.DataFrame(stage_metrics)
df_stages

## Part 2: Bottleneck Identification

Stages where average duration significantly exceeds the baseline average stage duration are flagged as potential bottlenecks.

In [ ]:
benchmark_avg = df_stages['avg_days'].mean()
print(f"Overall Average Stage Duration Benchmark: {benchmark_avg:.2f} days\n")

df_stages['is_bottleneck'] = df_stages['avg_days'] >= benchmark_avg
df_stages['severity'] = df_stages.apply(
    lambda r: 'High Bottleneck' if r['avg_days'] > benchmark_avg * 1.5
    else ('Medium Bottleneck' if r['is_bottleneck'] else 'Normal'),
    axis=1
)
df_stages[['stage', 'avg_days', 'median_days', 'max_days', 'is_bottleneck', 'severity']]

## Part 3: Time-to-Hire vs Time-to-Drop

We evaluate how quickly candidates transition through the pipeline based on their eventual outcome (Joined vs Dropped).

In [ ]:
hired = df[df['joined'] == 1]['total_recruitment_duration_days']
dropped = df[df['dropped'] == 1]['total_recruitment_duration_days']

outcome_df = pd.DataFrame([
    {
        'outcome': 'Hired & Joined',
        'candidate_count': len(hired),
        'avg_total_days': round(hired.mean(), 2),
        'median_total_days': round(hired.median(), 2),
        'min_days': hired.min(),
        'max_days': hired.max()
    },
    {
        'outcome': 'Dropped Candidates',
        'candidate_count': len(dropped),
        'avg_total_days': round(dropped.mean(), 2),
        'median_total_days': round(dropped.median(), 2),
        'min_days': dropped.min(),
        'max_days': dropped.max()
    }
])
outcome_df

## Part 4: Department & Role Velocity

We compare overall cycle time across departments to isolate departments experiencing slower turnaround.

In [ ]:
dept_velocity = []
for dept, grp in df.groupby('department'):
    h = grp[grp['joined'] == 1]['total_recruitment_duration_days']
    d = grp[grp['dropped'] == 1]['total_recruitment_duration_days']
    dept_velocity.append({
        'department': dept,
        'total_candidates': len(grp),
        'avg_pipeline_duration (days)': round(grp['total_recruitment_duration_days'].mean(), 2),
        'median_pipeline_duration (days)': round(grp['total_recruitment_duration_days'].median(), 2),
        'avg_time_to_hire (days)': round(h.mean(), 2) if not h.empty else None,
        'avg_time_to_drop (days)': round(d.mean(), 2) if not d.empty else None,
        'delayed_dropoffs_count': int(grp['is_delayed_dropoff'].sum()) if 'is_delayed_dropoff' in grp.columns else 0
    })

df_dept_vel = pd.DataFrame(dept_velocity).sort_values(by='avg_pipeline_duration (days)', ascending=False)
df_dept_vel